# E1 Stage 1 cycle S1c — B4, simulator only, guard absent by omission (2026-09-15)

Code `/Users/terrancehamilton/reachy-1-2-sim-stage1` = `main` `6bca12e`. Legs: `setup_c` PLACE_ROUTE via `rig_motion.deploy_to_rest` (HOME->REST), `flight_c` LIFT_TO_PRESENT via `rig_motion.to_present` (REST->PRESENT). One attempt each; a `stop` marker or a failed start check means no motion.

In [ ]:
# Cell 1 — connect with literals; hygiene shown
import os, sys, time, json, pathlib, traceback
sys.path.insert(0, "/Users/terrancehamilton/reachy-1-2-sim-stage1/src"); sys.path.insert(0, "/Users/terrancehamilton/reachy-1-2-sim-stage1/scripts")
CTRL = pathlib.Path("/Users/terrancehamilton/reachy-1-2-sim-stage1/docs/reviews/probes-2026-09-15-e1-stage1-b4/control"); RECORD_ROOT = "/Users/terrancehamilton/reachy-1-2-sim-stage1/docs/reviews/probes-2026-09-15-e1-stage1-b4/e1_server_runs"
SCENE = "/Users/terrancehamilton/reachy-1-2-sim-stage1/scenes/e1_boards/B4_pool_box_1_r2c3.yaml"; LEAD_IN_S = 3.0; CYCLE = "S1c"
print("REACHY env in this kernel:", {k: v for k, v in os.environ.items() if k.upper().startswith("REACHY")})
assert "REACHY_IP" not in os.environ and "REACHY_ENABLE_MOTION" not in os.environ, "shell hygiene violated"
from reachy_sdk import ReachySDK
HOST, PORT = "localhost", 50051
reachy = ReachySDK(host=HOST, sdk_port=PORT)
print(f"ReachySDK(host={HOST!r}, sdk_port={PORT}) connected at wall {time.time_ns()} mono {time.monotonic_ns()}")
print("python:", sys.executable)


In [ ]:
# Cell 2 — motion-client binding check: the recorder's identity function on THIS SDK object
import e1_identity
from reachy_ai.motion import rig_routes as R
from reachy_ai.motion import primitives
from reachy_ai.tasks import rig_motion
def _pose(): return {name: float(getattr(reachy.r_arm, name).present_position) for name in R.R_JOINTS}
ident = e1_identity.verify_simulator_identity(host=HOST, port=PORT, scene_path=SCENE, record_root=RECORD_ROOT, read_sdk_joints=_pose)
d = ident.as_dict(); print(json.dumps(d, indent=2, default=str))
BINDING_OK = bool(ident.ok)
(CTRL / (f"binding_ok_{CYCLE}" if BINDING_OK else f"binding_FAIL_{CYCLE}")).write_text(json.dumps(d, default=str))
print("BINDING_OK =", BINDING_OK); print("present:", {k: round(v, 1) for k, v in _pose().items()})
def wait_for(pred, timeout_s, period=0.25):
    t0 = time.monotonic()
    while time.monotonic() - t0 < timeout_s:
        if (CTRL / "stop").exists(): return "stop"
        if pred(): return "ready"
        time.sleep(period)
    return "timeout"
def start_check(kind):
    p = _pose(); here = R.posture_of(p)
    if kind == "PLACE_ROUTE_start": ok, why = rig_motion.check_start(reachy.r_arm, R.PLACE_ROUTE)
    elif kind == "PRESENT": ok = R.at_pose(p, R.PRESENT, tol=12.0, joints=list(R.GROSS_JOINTS)); why = "" if ok else "not at PRESENT (gross, 12 deg)"
    elif kind == "REST": ok = R.at_pose(p, R.REST, tol=12.0, joints=list(R.GROSS_JOINTS)); why = "" if ok else "not at REST (gross, 12 deg)"
    return {"kind": kind, "ok": bool(ok), "why": why, "posture_of": here, "pose": {k: round(v, 1) for k, v in p.items()}}
PREV_OK = BINDING_OK


In [ ]:
# Leg setup_c: PLACE_ROUTE via rig_motion.deploy_to_rest (HOME->REST) — one attempt, gated on go_setup_c + the recorder
LEG = {"leg": "setup_c", "route": "PLACE_ROUTE", "tool": "rig_motion.deploy_to_rest", "cycle": CYCLE}
go = wait_for(lambda: (CTRL / "go_setup_c").exists(), 1800) if PREV_OK else "not_eligible"
LEG["go"] = go; print("go:", go)
rec = wait_for(lambda: (CTRL / "recorder_setup_c.log").exists() and "fly the route now" in (CTRL / "recorder_setup_c.log").read_text(), 900) if go == "ready" else go
LEG["recorder_status"] = rec; print("recorder status:", rec)
if rec == "ready":
    time.sleep(LEAD_IN_S)
    LEG["start_check"] = start_check("PLACE_ROUTE_start"); print("start check:", LEG["start_check"])
if rec == "ready" and LEG["start_check"]["ok"]:
    LEG["t_start_mono_ns"] = time.monotonic_ns(); LEG["t_start_wall_ns"] = time.time_ns()
    phases = []
    reachy.turn_on("r_arm")
    try:
        ret = rig_motion.deploy_to_rest(reachy.r_arm, on_phase=lambda *a: phases.append([time.monotonic_ns(), *map(str, a)]))
        LEG["outcome"] = "returned"; LEG["returned"] = ret
    except Exception as exc:
        LEG["outcome"] = f"EXC {type(exc).__name__}: {exc}"; traceback.print_exc()
    LEG["phases"] = phases
    LEG["t_end_mono_ns"] = time.monotonic_ns(); LEG["t_end_wall_ns"] = time.time_ns()
    LEG["elapsed_s"] = (LEG["t_end_mono_ns"] - LEG["t_start_mono_ns"]) / 1e9
    LEG["end_pose"] = {k: round(v, 1) for k, v in _pose().items()}
    print("outcome:", LEG["outcome"], "elapsed %.1f s" % LEG["elapsed_s"]); print("returned:", LEG.get("returned")); print("end pose:", LEG["end_pose"])
else:
    LEG["outcome"] = "not_attempted"
PREV_OK = LEG["outcome"] == "returned"
(CTRL / "setup_c_done").write_text(json.dumps(LEG)); print(json.dumps(LEG, default=str))


In [ ]:
# Leg flight_c: LIFT_TO_PRESENT via rig_motion.to_present (REST->PRESENT) — one attempt, gated on go_flight_c + the recorder
LEG = {"leg": "flight_c", "route": "LIFT_TO_PRESENT", "tool": "rig_motion.to_present", "cycle": CYCLE}
go = wait_for(lambda: (CTRL / "go_flight_c").exists(), 1800) if PREV_OK else "not_eligible"
LEG["go"] = go; print("go:", go)
rec = wait_for(lambda: (CTRL / "recorder_flight_c.log").exists() and "fly the route now" in (CTRL / "recorder_flight_c.log").read_text(), 900) if go == "ready" else go
LEG["recorder_status"] = rec; print("recorder status:", rec)
if rec == "ready":
    time.sleep(LEAD_IN_S)
    LEG["start_check"] = start_check("REST"); print("start check:", LEG["start_check"])
if rec == "ready" and LEG["start_check"]["ok"]:
    LEG["t_start_mono_ns"] = time.monotonic_ns(); LEG["t_start_wall_ns"] = time.time_ns()
    phases = []
    reachy.turn_on("r_arm")
    try:
        ret = rig_motion.to_present(reachy.r_arm, on_phase=lambda *a: phases.append([time.monotonic_ns(), *map(str, a)]))
        LEG["outcome"] = "returned"; LEG["returned"] = ret
    except Exception as exc:
        LEG["outcome"] = f"EXC {type(exc).__name__}: {exc}"; traceback.print_exc()
    LEG["phases"] = phases
    LEG["t_end_mono_ns"] = time.monotonic_ns(); LEG["t_end_wall_ns"] = time.time_ns()
    LEG["elapsed_s"] = (LEG["t_end_mono_ns"] - LEG["t_start_mono_ns"]) / 1e9
    LEG["end_pose"] = {k: round(v, 1) for k, v in _pose().items()}
    print("outcome:", LEG["outcome"], "elapsed %.1f s" % LEG["elapsed_s"]); print("returned:", LEG.get("returned")); print("end pose:", LEG["end_pose"])
else:
    LEG["outcome"] = "not_attempted"
PREV_OK = LEG["outcome"] == "returned"
(CTRL / "flight_c_done").write_text(json.dumps(LEG)); print(json.dumps(LEG, default=str))


In [ ]:
# Final read-only state; no further motion
print("final pose:", {k: round(v, 1) for k, v in _pose().items()}); print("done at wall", time.time_ns())
